In [ ]:
!pip install polars

In [ ]:
# Initialize Hail without setting default reference
import polars as pl
import hail as hl
hl.init()

# Set the default reference genome after initialization
hl.default_reference('GRCh38')

In [ ]:
!gcloud storage ls gs://vwb-aou-allxall/v8

In [ ]:
!gcloud storage ls gs://vwb-aou-allxall/v8/utility_ht/

In [ ]:
VAR_METADATA = "gs://vwb-aou-allxall/v8/utility_ht/aou_exome_variant_qc_annotated.ht"

vep_ht = hl.read_table(VAR_METADATA)
vep_ht.show(5)

In [ ]:
unique_filters = vep_ht.aggregate(
    hl.agg.explode(lambda x: hl.agg.collect_as_set(x), sampled.filters)
)

In [ ]:
sampled = vep_ht.sample(0.01)
unique_filters = sampled.aggregate(
    hl.agg.explode(lambda x: hl.agg.collect_as_set(x), sampled.filters)
)
print(unique_filters)


In [ ]:
vep_ht.filters.show(10)


In [ ]:
sampled.aggregate(
    hl.agg.count_where(
        hl.is_defined(sampled.filters) & (hl.len(sampled.filters) > 0)
    )
)

# QC validation — does `aou_exome_variant_qc_annotated.ht` carry the paper's QC?

Strategy: independently re-derive the paper's QC from the **raw exome split MT**
(entry-level `GT/GQ/AD/FT`) and compare per-variant `AC`, `call_rate`, `gq_stats`
to the all-by-all HT, scoped to a small interval (cheap; widen once it reconciles).

**Paper QC**
- *Variant-level*: passed VQSR/VETS (`FT`); global `AC > 0`
- *Genotype-level ("adj")*: `GQ >= 30`; minor allele balance `> 0.2` for hets;
  drop chrY genotypes for self-reported-female samples

**Confirmed in this dataset**
- exome MT entry: `{GQ, PS, RGQ, FT(str 'PASS'/'FAIL'/missing), AD, GT}`
- sample-meta `sex`: **0 = female, 1 = male** (verified empirically on chrY)
- site `filters` is empty in AoU — VETS lives per-allele in `as_vets`
  (`calibration_sensitivity`) in the all-by-all HT only.

In [ ]:
# Enable requester-pays on the *already running* Spark context (controlled-tier bucket).
# No hl.stop()/restart needed.
BILLING = os.environ["GOOGLE_CLOUD_BILLING_PROJECT"]
_hconf = hl.spark_context()._jsc.hadoopConfiguration()
_hconf.set("fs.gs.requester.pays.mode", "AUTO")
_hconf.set("fs.gs.requester.pays.project.id", BILLING)

EXOME_MT = "gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/splitMT/hail.mt"
META_HT  = "gs://vwb-aou-allxall/v8/utility_ht/aou_v8_final_sample_meta.ht"
ALLXALL  = "gs://vwb-aou-allxall/v8/utility_ht/aou_exome_variant_qc_annotated.ht"

# Start tiny; widen once the numbers reconcile.
TEST_INTERVAL = "chr1:55039000-55064000"   # ~PCSK9, a handful of variants

In [ ]:
# 1. raw exome MT restricted to the test region, with sample sex joined in
mt = hl.read_matrix_table(EXOME_MT)
mt = hl.filter_intervals(mt, [hl.parse_locus_interval(TEST_INTERVAL)])
meta = hl.read_table(META_HT)
mt = mt.annotate_cols(sex=meta[mt.s].sex)   # 0 = female, 1 = male
mt.describe()

In [ ]:
# 2. the paper's genotype-level "adj" filter  (TUNABLE — see notes cell below)
# Split MT => biallelic rows: AD = [ref_depth, alt_depth]
ab = mt.AD[1] / hl.sum(mt.AD)

# FT == 'PASS' constrains only *variant* (non-ref) genotypes; hom-ref / ref blocks
# carry FT = missing and must NOT be dropped by the VQSR restriction.
ft_ok = (~mt.GT.is_non_ref()) | (mt.FT == "PASS")

adj = (
    ft_ok
    & (mt.GQ >= 30)
    & hl.if_else(mt.GT.is_het(), ab > 0.2, True)        # AB only for hets
    & ~((mt.locus.contig == "chrY") & (mt.sex == 0))    # drop chrY in females
)

mt_qc = hl.variant_qc(mt.filter_entries(adj))
recomputed = mt_qc.rows().select(
    my_AC        = mt_qc.variant_qc.AC[1],
    my_AN        = mt_qc.variant_qc.AN,
    my_call_rate = mt_qc.variant_qc.call_rate,
    my_n_filt    = mt_qc.variant_qc.n_filtered,
    my_gq_mean   = mt_qc.variant_qc.gq_stats.mean,
    my_n_het     = mt_qc.variant_qc.n_het,
)

In [ ]:
# 3. all-by-all HT for the same region, then compare per variant
axa = hl.read_table(ALLXALL)
axa = hl.filter_intervals(axa, [hl.parse_locus_interval(TEST_INTERVAL)])
axa = axa.select(
    axa_AC        = axa.info.AC[axa.a_index - 1],
    axa_call_rate = axa.variant_qc.call_rate,
    axa_n_filt    = axa.variant_qc.n_filtered,
    axa_gq_mean   = axa.variant_qc.gq_stats.mean,
    axa_n_het     = axa.variant_qc.n_het,
)

cmp = recomputed.join(axa, how="inner")
cmp = cmp.annotate(
    AC_match = cmp.my_AC == cmp.axa_AC,
    cr_match = hl.abs(cmp.my_call_rate - cmp.axa_call_rate) < 1e-6,
    gq_match = hl.abs(cmp.my_gq_mean   - cmp.axa_gq_mean)   < 1e-3,
)
cmp.show(50)

In [ ]:
# 4. agreement summary over the test region
cmp.aggregate(hl.struct(
    n        = hl.agg.count(),
    AC_match = hl.agg.fraction(cmp.AC_match),
    cr_match = hl.agg.fraction(cmp.cr_match),
    gq_match = hl.agg.fraction(cmp.gq_match),
))

### Tuning notes

If `AC`/`call_rate`/`gq_stats` match → the all-by-all HT carries exactly this QC.
If they don't, adjust the **adj** definition in step 2 and re-run — the goal is to
find the definition that reproduces the HT's `n_filtered`/`call_rate`/`AC`:

- **FT on hom-ref**: this version keeps hom-ref genotypes regardless of FT. If the HT
  was built by dropping *all* non-PASS-FT entries, change `ft_ok` to `mt.FT == "PASS"`.
- **Allele balance**: "minor allele balance" may mean `min(AD)/sum(AD) > 0.2`
  (filters both ref- and alt-skewed hets) rather than `AD[1]/sum(AD) > 0.2`.
  Try `mab = hl.min(mt.AD) / hl.sum(mt.AD)`.
- **DP**: gnomAD adj also uses `DP >= 10`; the paper omits it. There is no `DP`
  entry here (only `AD`), so leave it out unless AC mismatches systematically.

**Variant-level VETS** (separate from the genotype filter): the all-by-all HT stores
it per-allele in `as_vets` (`{model, calibration_sensitivity}`), not in `filters`.
To check the VQSR-pass set, threshold `calibration_sensitivity` per model, e.g.:

```python
axa = hl.read_table(ALLXALL)
axa.aggregate(hl.agg.explode(
    lambda kv: hl.agg.group_by(kv[1].model, hl.agg.stats(kv[1].calibration_sensitivity)),
    hl.array(axa.as_vets)))
```

To **widen** the validation, set `TEST_INTERVAL` to a whole small contig
(e.g. `"chr21"`) once a single region reconciles.